# Task 3 — A/B Hypothesis Testing for ACIS

This notebook tests the Task 3 hypotheses using the ACIS insurance dataset.  
The analysis works on a policy-level snapshot so that claim frequency, claim severity, and margin are evaluated at the policy/coverage level rather than repeatedly counting the same monthly row.

## 1) Imports and reusable functions

The dataset columns include policy identifiers, geography, vehicle features, and financial outcomes such as `TotalPremium` and `TotalClaims`. We load the raw pipe-separated file, prepare a policy-level view, then run the statistical tests.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

from data_loader import load_insurance_data
from preprocessing import prepare_policy_level_data, normalize_gender
from hypothesis_tests import (
    matched_segment,
    chi_square_claim_frequency_test,
    welch_ttest_numeric,
    build_result_row,
    decision_from_pvalue,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

## 2) Load and prepare the data

The task data contains fields such as `UnderwrittenCoverID`, `PolicyID`, `TransactionMonth`, `Province`, `PostalCode`, `VehicleType`, `TotalPremium`, and `TotalClaims`. We read the file with `sep='|'` and build a policy-level frame.

In [ ]:
DATA_PATH = Path("../data/insurance_data.csv")

df_raw = load_insurance_data(DATA_PATH)
df_raw = normalize_gender(df_raw)

policy = prepare_policy_level_data(df_raw)

print("Raw rows:", len(df_raw))
print("Policy-level rows:", len(policy))
policy.head()

## 3) Derived metrics used in Task 3

- **Claimed**: whether a policy had at least one claim.
- **Margin**: `TotalPremium - TotalClaims`

These are the base variables for claim frequency, claim severity, and profit/margin testing.

In [ ]:
# Quick snapshot
summary = pd.DataFrame({
    "Metric": [
        "Total premium",
        "Total claims",
        "Claim frequency",
        "Average claim severity (positive claims only)",
        "Average margin"
    ],
    "Value": [
        f"R {policy['TotalPremium'].sum():,.2f}",
        f"R {policy['TotalClaims'].sum():,.2f}",
        f"{policy['Claimed'].mean() * 100:.2f}%",
        f"R {policy.loc[policy['TotalClaims'] > 0, 'TotalClaims'].mean():,.2f}",
        f"R {policy['Margin'].mean():,.2f}"
    ]
})
summary

## 4) Hypothesis 1 — No risk differences across provinces

**KPI:** Claim Frequency  
**Test:** Chi-squared test of independence

We compare two major provinces to keep the analysis interpretable and to avoid unequal segment sizes. The code below selects the two largest provinces and then matches the sample on the most common vehicle and cover segment shared by both groups.

In [ ]:
province_a, province_b = policy["Province"].value_counts().index[:2].tolist()

province_subset = matched_segment(
    policy,
    group_col="Province",
    group_a=province_a,
    group_b=province_b,
    match_cols=["VehicleType", "CoverType"]
)

province_result = chi_square_claim_frequency_test(
    province_subset,
    group_col="Province",
    group_a=province_a,
    group_b=province_b,
    kpi_col="Claimed"
)

province_summary = (
    province_subset.groupby("Province")
    .agg(
        policies=("UnderwrittenCoverID", "count"),
        claim_frequency=("Claimed", "mean"),
        avg_severity=("TotalClaims", lambda s: s[s > 0].mean()),
        avg_margin=("Margin", "mean")
    )
    .reset_index()
)

province_summary

In [ ]:
province_result

## 5) Hypothesis 2 — No risk differences between zip codes

**KPI:** Claim Severity  
**Test:** Welch's t-test

We compare two high-volume postal codes inside the same province-level segment. This keeps the comparison focused on geography rather than product mix.

In [ ]:
# Focus on the most common province first to reduce confounding
top_province = policy["Province"].value_counts().index[0]
province_for_zip = policy[policy["Province"] == top_province].copy()

zip_a, zip_b = province_for_zip["PostalCode"].value_counts().index[:2].tolist()

zip_subset = matched_segment(
    province_for_zip,
    group_col="PostalCode",
    group_a=zip_a,
    group_b=zip_b,
    match_cols=["VehicleType", "CoverType"]
)

zip_severity_result = welch_ttest_numeric(
    zip_subset,
    group_col="PostalCode",
    group_a=zip_a,
    group_b=zip_b,
    value_col="TotalClaims",
    positive_only=True
)

zip_summary = (
    zip_subset.groupby("PostalCode")
    .agg(
        policies=("UnderwrittenCoverID", "count"),
        claim_frequency=("Claimed", "mean"),
        avg_severity=("TotalClaims", lambda s: s[s > 0].mean()),
        avg_margin=("Margin", "mean")
    )
    .reset_index()
)

zip_summary

In [ ]:
zip_severity_result

## 6) Hypothesis 3 — No significant margin difference between zip codes

**KPI:** Margin  
**Test:** Welch's t-test

This uses the same postal-code comparison, but the target metric is margin rather than claim severity.

In [ ]:
zip_margin_result = welch_ttest_numeric(
    zip_subset,
    group_col="PostalCode",
    group_a=zip_a,
    group_b=zip_b,
    value_col="Margin",
    positive_only=False
)

zip_margin_result

## 7) Hypothesis 4 — No significant risk difference between Women and Men

**KPI:** Claim Frequency  
**Test:** Chi-squared test of independence

We normalize the gender labels, keep the male/female records, and test whether claim frequency differs by gender.

In [ ]:
gender_subset = policy[policy["Gender"].isin(["Female", "Male"])].copy()

gender_result = chi_square_claim_frequency_test(
    gender_subset,
    group_col="Gender",
    group_a="Female",
    group_b="Male",
    kpi_col="Claimed"
)

gender_summary = (
    gender_subset.groupby("Gender")
    .agg(
        policies=("UnderwrittenCoverID", "count"),
        claim_frequency=("Claimed", "mean"),
        avg_severity=("TotalClaims", lambda s: s[s > 0].mean()),
        avg_margin=("Margin", "mean")
    )
    .reset_index()
)

gender_summary

In [ ]:
gender_result

## 8) Results table

The table below combines the hypotheses, the KPI used, the test used, the p-value, and the decision rule (`p < 0.05` ⇒ reject the null hypothesis).

In [ ]:
results = pd.DataFrame([
    build_result_row(
        hypothesis="No risk differences across provinces",
        kpi="Claim Frequency",
        test_name="Chi-squared",
        p_value=province_result["p_value"],
    ),
    build_result_row(
        hypothesis="No risk differences between zip codes",
        kpi="Claim Severity",
        test_name="Welch t-test",
        p_value=zip_severity_result["p_value"],
    ),
    build_result_row(
        hypothesis="No significant margin difference between zip codes",
        kpi="Margin",
        test_name="Welch t-test",
        p_value=zip_margin_result["p_value"],
    ),
    build_result_row(
        hypothesis="No significant risk difference between Women and Men",
        kpi="Claim Frequency",
        test_name="Chi-squared",
        p_value=gender_result["p_value"],
    ),
])

results

## 9) Business-facing interpretation template

After you run the notebook on the real ACIS dataset, use the decisions in the results table to write short business recommendations for each rejected hypothesis, for example:
- premium adjustments for high-risk provinces,
- location-specific pricing changes for high-loss postal codes,
- and gender-neutral or gender-sensitive underwriting rules only where statistically justified.